<a href="https://colab.research.google.com/github/jeremydouti2-hub/Data-Science-Projects/blob/main/Customer_Satisfaction_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Kaggle
!pip install kaggle

# Upload kaggle.json file (API key)
from google.colab import files
files.upload()

# Configure Kaggle credentials
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Download dataset from Kaggle
!kaggle datasets download -d jessemostipak/hotel-booking-demand

# Extract the ZIP file
!unzip hotel-booking-demand.zip

In [ ]:
import pandas as pd

data = pd.read_csv("hotel_bookings.csv")

data.head()

In [ ]:
data.info()
data.isnull().sum()
data.duplicated().sum()

In [ ]:
# --- Dataset Structure ---
print(data.info())

# --- Missing Values ---
print(data.isnull().sum())

# --- Duplicate Rows ---
print("Number of duplicates:", data.duplicated().sum())

# --- Statistical Summary ---
print(data.describe())

# --- Distribution Plot ---
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(data['adr'])  # Average Daily Rate (ADR)
plt.show()

In [ ]:
# Remove duplicate rows
data = data.drop_duplicates()

print("Remaining duplicates:", data.duplicated().sum())

In [ ]:
# Missing values
data.isnull().sum()

In [ ]:
# apply cleaning
# Drop column with too many missing values
data = data.drop(columns=['company'])

# Fill numerical column with median
data['agent'].fillna(data['agent'].median(), inplace=True)

# Fill categorical column with mode
data['country'].fillna(data['country'].mode()[0], inplace=True)

In [ ]:
# Convert to correct types
data['children'] = pd.to_numeric(data['children'], errors='coerce')

# Fill any new NaN created
data['children'].fillna(data['children'].median(), inplace=True)

In [ ]:
# Clean categorical text columns
text_cols = ['hotel', 'meal', 'market_segment', 'distribution_channel', 'customer_type', 'country']

for col in text_cols:
    data[col] = data[col].astype(str).str.strip().str.title()

In [ ]:
# Cap extreme values
data['adr'] = data['adr'].clip(lower=0, upper=data['adr'].quantile(0.99))

In [ ]:
# Create total guests
data['total_guests'] = data['adults'] + data['children'] + data['babies']

# Create stay duration
data['total_nights'] = data['stays_in_weekend_nights'] + data['stays_in_week_nights']

In [ ]:
print("Missing values after cleaning:\n", data.isnull().sum())
print("Duplicates after cleaning:", data.duplicated().sum())

data.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(data['adr'], bins=30, kde=True)
plt.title("Distribution of Average Daily Rate (ADR)")
plt.xlabel("ADR")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sns.boxplot(x='customer_type', y='adr', data=data)
plt.title("ADR by Customer Type")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(data.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
sns.scatterplot(x='total_guests', y='adr', data=data)
plt.title("Total Guests vs ADR")
plt.show()

In [ ]:
data.groupby('hotel')['adr'].mean().plot(kind='bar')
plt.title("Average ADR by Hotel Type")
plt.ylabel("ADR")
plt.show()

In [ ]:
data[['adr', 'total_guests', 'total_nights']].std()

In [ ]:
# Select important numerical variables
corr_matrix = data[['adr', 'total_guests', 'total_nights',
                    'adults', 'children', 'babies']].corr()

print(corr_matrix)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Key Correlations")
plt.show()

In [ ]:
# Features (independent variables)
X = data[['total_guests', 'total_nights', 'adults', 'children']]

# Target (dependent variable)
y = data['adr']

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate model
r2 = model.score(X_test, y_test)

print("R-squared:", r2)

In [ ]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

print(importance)